# Experimentos de Pi-zero (pi0) sobre LIBERO

Corre pi0 (LeRobot) en varias tareas de LIBERO y muestra los resultados (tabla,
tasa de éxito y videos). Es el mismo framework que el notebook de OpenVLA
(`core`, `benchmarks`, `models`); aquí solo cambia el **modelo**.

> Requiere el entorno `pizero` (ver `environment-pizero.yml`), con **GPU** y el
> checkpoint `lerobot/pi0_libero_finetuned` descargado en `HF_HOME`.


In [1]:
!pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu118
#!pip install "lerobot[pi]@git+https://github.com/huggingface/lerobot.git"

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.7/811.7 MB 24.3 MB/s  0:00:15:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 14.0 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 35.7 MB/s  0:00:00m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 2.7 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 26.2 MB/s  0:00:00m0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 26.1 MB/s  0:00:14:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 37.3 MB/s  0:00:08:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 41.7 MB/s  0:00:04:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 39.3 MB/s  0:00:01m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 42.9 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install "numpy<2"

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 22.2 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  You can safely remove it manually.
  You can safely remove it manually.


In [ ]:
!pip install "lerobot[pi]==0.4.0"

  Using cached lerobot-0.4.0-py3-none-any.whl.metadata (25 kB)
  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
  Using cached diffusers-0.35.2-py3-none-any.whl.metadata (20 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached setuptools-80.10.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached cmake-4.1.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.5 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached av-15.1.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (4.6 kB)
  Using cached jsonlines-4.0.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pynput-1.8.2-py2.py3-none-any.whl.metadata (32 kB)
  Using cached pyserial-3.5-py2.py3-

## 1. Setup


In [6]:
import os, sys, glob

os.environ.setdefault("MUJOCO_GL", "egl")     # render headless (sin pantalla)

# --- Shim de torch.load (necesario con GPUs nuevas) ---
# Con GPU nueva (Blackwell) se necesita torch >= 2.6, cuyo torch.load usa por
# defecto weights_only=True; eso rompe la carga de checkpoints (pickles con
# numpy). Restauramos el comportamiento anterior. Seguro aqui: los checkpoints
# son de fuente confiable. Debe ir ANTES de cargar modelo o entorno.
import functools
import torch
torch.load = functools.partial(torch.load, weights_only=False)


def _find_project_root():
    """Sube desde el cwd buscando la raiz del proyecto (simulation.py + core/)."""
    d = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, "simulation.py")) and \
           os.path.isdir(os.path.join(d, "core")):
            return d
        d = os.path.dirname(d)
    hits = glob.glob(os.path.join(os.getcwd(), "**", "simulation.py"), recursive=True)
    if hits:
        return os.path.dirname(os.path.abspath(hits[0]))
    raise RuntimeError("No encontre la raiz del proyecto (simulation.py + core/).")


ROOT = _find_project_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", os.path.join(ROOT, "hf_cache"))   # cache de pesos
print("Proyecto:", ROOT)
print("HF_HOME :", os.environ["HF_HOME"])


Proyecto: /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation
HF_HOME : /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/hf_cache


## 2. Configuración


In [8]:
from dataclasses import dataclass


@dataclass
class Config:
    suite: str = "libero_10"
    num_tasks: int = 3             # cuantos escenarios (tareas) probar
    episodes_per_task: int = 1     # configuraciones iniciales por tarea
    max_steps: int = 520           # libero_10 es de horizonte largo (oficial ~520)
    out_dir: str = "output/experiments_pizero"


cfg = Config()
cfg


Config(suite='libero_10', num_tasks=3, episodes_per_task=1, max_steps=520, out_dir='output/experiments_pizero')

## 3. Cargar el modelo (una sola vez)


In [7]:
from core import View
from models.Pi_zero import PiZeroController

model = PiZeroController(view=View.AGENT, device="cuda")
print("modelo pi0 cargado")


ModuleNotFoundError: No module named 'lerobot'

## 4. Correr los experimentos
Un `LiberoController` por tarea (escenario); `run_experiments` corre los
episodios y graba un video por cada uno. pi0 devuelve un chunk de acciones que
el runner ejecuta una a una.


In [ ]:
from core import run_experiments
from benchmarks.libero import LiberoController

task_list = LiberoController.tasks(cfg.suite)[:cfg.num_tasks]
benchmarks = [LiberoController(task_id=tid, suite=cfg.suite) for tid, _ in task_list]

results = run_experiments(
    model, benchmarks,
    max_steps=cfg.max_steps,
    episodes_per_task=cfg.episodes_per_task,
    out_dir=cfg.out_dir,
    record_view=View.AGENT,
)
print(f"{len(results)} episodios corridos")


## 5. Resultados — tabla y tasa de éxito


In [ ]:
import pandas as pd
from core import summarize

df = pd.DataFrame(results)
resumen = summarize(results)
print("Tasa de exito global: {exitos}/{total} = {tasa_exito:.0%}".format(**resumen))
df[["scenario", "instruction", "episode", "success", "steps"]]


In [ ]:
import matplotlib.pyplot as plt

por_escenario = df.groupby("scenario")["success"].mean()
ax = por_escenario.plot(kind="bar", ylim=(0, 1), color="#F58518", rot=0)
ax.set_xlabel("escenario"); ax.set_ylabel("tasa de exito")
ax.set_title("Exito por tarea (pi0)"); plt.tight_layout(); plt.show()


## 6. Resultados — videos


In [ ]:
from IPython.display import Video, display

for r in results:
    if r["video"]:
        print(f"escenario {r['scenario']} | {r['instruction']} | exito={r['success']}")
        display(Video(r["video"], embed=True, width=320))
